In [ ]:
#@title Setup (run this first)

!pip install transformers --quiet

import torch
import torch.nn.functional as F
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import warnings
warnings.filterwarnings('ignore')

print("Loading GPT-2... ", end="")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Done!")

# Global state
_current_text = ""
_current_options = {}

def _get_top_4_tokens(text):
    """Get the top 4 most likely next tokens."""
    inputs = tokenizer.encode(text, return_tensors='pt')
    with torch.no_grad():
        outputs = model(inputs)
    logits = outputs.logits[0, -1, :]
    probs = F.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, 4)

    tokens = {}
    for i, letter in enumerate(['A', 'B', 'C', 'D']):
        token_id = top_indices[i].item()
        token_text = tokenizer.decode(token_id)
        tokens[letter] = {'id': token_id, 'text': token_text}
    return tokens

def _display_options(text, options):
    """Display the current text and options in big format."""
    display_text = text.upper() + " ___"

    print()
    print("=" * 64)
    print(display_text)
    print("=" * 64)
    print()

    # Get token texts and find max width
    texts = [options[l]['text'] for l in ['A', 'B', 'C', 'D']]
    max_width = max(len(t) for t in texts)
    box_width = max(max_width + 2, 8)  # minimum 8 chars wide

    # Header row with letters
    header = ""
    for letter in ['A', 'B', 'C', 'D']:
        header += letter.center(box_width + 4)
    print(header)

    # Top border
    border = ""
    for _ in ['A', 'B', 'C', 'D']:
        border += ("+" + "-" * box_width + "+").center(box_width + 4)
    print(border)

    # Token text row
    content = ""
    for letter in ['A', 'B', 'C', 'D']:
        token_text = options[letter]['text']
        content += ("|" + token_text.center(box_width) + "|").center(box_width + 4)
    print(content)

    # Bottom border
    print(border)

    print()
    print("Move to your corner!")
    print()

def start_sentence(text):
    """Start a new sentence and show options."""
    global _current_text, _current_options
    _current_text = text
    _current_options = _get_top_4_tokens(text)
    _display_options(_current_text, _current_options)

def pick_token(choice):
    """Pick a token and show next options."""
    global _current_text, _current_options
    choice = choice.upper()
    if choice not in _current_options:
        print("Please choose A, B, C, or D")
        return

    chosen_token = _current_options[choice]['text']
    _current_text = _current_text + chosen_token
    _current_options = _get_top_4_tokens(_current_text)
    _display_options(_current_text, _current_options)

def show_final():
    """Display the final sentence."""
    print()
    print("=" * 64)
    print("FINAL SENTENCE:")
    print(f'"{_current_text}"')
    print("=" * 64)
    print()

print()
print("Ready for the Human Sampler activity.")

In [ ]:
#@title Start a new sentence
starting_text = "" #@param {type:"string"}

start_sentence(starting_text)

In [ ]:
#@title Pick the winning token
choice = "A" #@param ["A", "B", "C", "D"]

pick_token(choice)

In [ ]:
#@title Show final sentence

show_final()